In [1]:
import numpy as np
from pathlib import Path

training_dir = Path("/p/scratch/training2600/TeamGudnason/training_data")

# Temporal split
TRAIN_FILES = [
    "patches_S2B_MSIL2A_20180408T095029_N0500_R079_T33TYN_20230906T162829_data.npz",
    "patches_S2A_MSIL2A_20180523T095031_N0500_R079_T33TYN_20230903T233859_data.npz",
]
VAL_FILES = [
    "patches_S2B_MSIL2A_20180816T095029_N0500_R079_T33TYN_20230812T013012_data.npz",
]
TEST_FILES = [
    "patches_S2A_MSIL2A_20181020T095031_N0500_R079_T33TYN_20230728T230408_data.npz",
]

def load_files(file_list):
    all_X, all_y = [], []
    for fname in file_list:
        d = np.load(training_dir / fname)
        all_X.append(d["patches"])
        all_y.append(d["labels"])
    return np.concatenate(all_X), np.concatenate(all_y)

X_train_raw, y_train_raw = load_files(TRAIN_FILES)
X_val_raw, y_val_raw     = load_files(VAL_FILES)
X_test_raw, y_test_raw   = load_files(TEST_FILES)

print(f"Train: {X_train_raw.shape}, {np.unique(y_train_raw)}")
print(f"Val:   {X_val_raw.shape}")
print(f"Test:  {X_test_raw.shape}")

Train: (100000, 3, 3, 4), [ 2  3  7 10 11 12 15 18 20 21 23 25 29 35 40 41]
Val:   (50000, 3, 3, 4)
Test:  (50000, 3, 3, 4)


In [2]:
# Preprocess
X_train_raw = X_train_raw.astype(np.float32) * 0.0001
X_val_raw   = X_val_raw.astype(np.float32) * 0.0001
X_test_raw  = X_test_raw.astype(np.float32) * 0.0001

X_train = X_train_raw.reshape(X_train_raw.shape[0], -1)
X_val   = X_val_raw.reshape(X_val_raw.shape[0], -1)
X_test  = X_test_raw.reshape(X_test_raw.shape[0], -1)

# Remap labels based on train set only
unique_labels = np.unique(y_train_raw)
label_to_idx = {int(lbl): i for i, lbl in enumerate(unique_labels)}
idx_to_label = {i: int(lbl) for i, lbl in enumerate(unique_labels)}

def remap(y):
    return np.array([label_to_idx.get(int(lbl), -1) for lbl in y], dtype=np.int64)

y_train = remap(y_train_raw)
y_val   = remap(y_val_raw)
y_test  = remap(y_test_raw)

# Drop classes not in train set and classes with < 200 samples
classes_tr, counts_tr = np.unique(y_train, return_counts=True)
valid_classes = set(classes_tr[counts_tr >= 200])

def filter_by_classes(X, y, valid):
    mask = np.isin(y, list(valid))
    return X[mask], y[mask]

X_train, y_train = filter_by_classes(X_train, y_train, valid_classes)
X_val,   y_val   = filter_by_classes(X_val,   y_val,   valid_classes)
X_test,  y_test  = filter_by_classes(X_test,  y_test,  valid_classes)

# Final remap to contiguous
unique_final = np.unique(y_train)
final_map = {int(lbl): i for i, lbl in enumerate(unique_final)}
idx_to_label = {i: int(unique_labels[int(lbl)]) for i, lbl in enumerate(unique_final)}

y_train = np.array([final_map[int(l)] for l in y_train], dtype=np.int64)
y_val   = np.array([final_map[int(l)] for l in y_val],   dtype=np.int64)
y_test  = np.array([final_map[int(l)] for l in y_test],  dtype=np.int64)

num_classes = len(unique_final)
print(f"Train: {X_train.shape[0]} samples, {num_classes} classes")
print(f"Val:   {X_val.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")

Train: 99638 samples, 13 classes
Val:   49819 samples
Test:  49819 samples


In [5]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import lightning.pytorch as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

class YourCustomDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

class ConvNet(pl.LightningModule):
    def __init__(self, num_classes=10, in_channels=4, patch_size=3,
                 lr=3e-4, max_epochs=60, class_weights=None):
        super().__init__()
        self.lr = lr
        self.in_channels = in_channels
        self.patch_size = patch_size
        self.num_classes = num_classes
        self.max_epochs = max_epochs
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128, 64),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )
        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def _to_image(self, x):
        x = x.float()
        h = w = self.patch_size
        c = self.in_channels
        if x.dim() == 2:
            x = x.view(x.size(0), h, w, c).permute(0, 3, 1, 2).contiguous()
        return x

    def forward(self, x):
        return self.classifier(self.features(self._to_image(x)))

    def _loss(self, logits, y):
        return F.cross_entropy(logits, y, weight=self.class_weights)

    def training_step(self, batch, _):
        x, y = batch
        loss = self._loss(self(x), y.long())
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        x, y = batch
        logits = self(x)
        loss = self._loss(logits, y.long())
        acc = (logits.argmax(1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def test_step(self, batch, _):
        x, y = batch
        logits = self(x)
        loss = self._loss(logits, y.long())
        acc = (logits.argmax(1) == y).float().mean()
        self.log("test_loss", loss)
        self.log("test_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.max_epochs)
        return {"optimizer": opt, "lr_scheduler": sched}

In [6]:
from torch.utils.data import DataLoader, WeightedRandomSampler
import torch

# Class weights
y_train_np = np.asarray(y_train, dtype=np.int64)
classes, counts = np.unique(y_train_np, return_counts=True)

raw_weights = np.zeros(len(classes), dtype=np.float64)
for c, n in zip(classes, counts):
    raw_weights[int(c)] = 1.0 / n
sqrt_weights = np.sqrt(raw_weights)
sqrt_weights /= sqrt_weights.sum() / len(classes)
class_weights_t = torch.tensor(sqrt_weights, dtype=torch.float32)

# Datasets
train_dataset = YourCustomDataset(X_train, y_train)
val_dataset   = YourCustomDataset(X_val,   y_val)
test_dataset  = YourCustomDataset(X_test,  y_test)

# Weighted sampler
sample_weights = np.array([1.0 / counts[int(l)] for l in y_train_np])
sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

train_dataloader = DataLoader(train_dataset, batch_size=256, sampler=sampler,
                               shuffle=False, num_workers=8, pin_memory=True,
                               persistent_workers=True)
val_dataloader   = DataLoader(val_dataset,   batch_size=256, shuffle=False,
                               num_workers=8, pin_memory=True, persistent_workers=True)
test_dataloader  = DataLoader(test_dataset,  batch_size=256, shuffle=False,
                               num_workers=8, pin_memory=True, persistent_workers=True)

# Model
patch_size  = 3
in_channels = X_train.shape[1] // (patch_size**2)
max_epochs  = 200

model_temporal = ConvNet(
    lr=3e-4,
    num_classes=num_classes,
    in_channels=in_channels,
    patch_size=patch_size,
    class_weights=None,
    max_epochs=max_epochs,
)

# Early stopping + trainer
early_stop = pl.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=15,
    mode="min",
)

trainer_temporal = pl.Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=max_epochs,
    gradient_clip_val=1.0,
    log_every_n_steps=50,
    enable_progress_bar=True,
    callbacks=[early_stop],
)

trainer_temporal.fit(model_temporal, train_dataloader, val_dataloader)
trainer_temporal.test(model_temporal, dataloaders=test_dataloader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3     ]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ features   │ Sequential │ 76.6 K │ train │     0 │
│ 1 │ classifier │ Sequential │  9.1 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 85.7 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 85.7 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 14                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

SLURM auto-requeueing enabled. Setting signal handlers.


Output()

/p/project1/training2600/TeamGudnason/envs/danube/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.
py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3     ]
SLURM auto-requeueing enabled. Setting signal handlers.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │   0.024709448218345642    │
│         test_loss         │    150.92617797851562     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 150.92617797851562, 'test_acc': 0.024709448218345642}]

In [7]:
model_temporal = ConvNet(
    lr=1e-4,           # lægra en áðan
    num_classes=num_classes,
    in_channels=in_channels,
    patch_size=patch_size,
    class_weights=class_weights_t,  # class weights í staðinn fyrir sampler
    max_epochs=max_epochs,
)

train_dataloader = DataLoader(
    train_dataset, batch_size=256,
    shuffle=True,       # shuffle í staðinn fyrir sampler
    num_workers=8, pin_memory=True, persistent_workers=True
)

early_stop = pl.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=20,        # meira patience
    mode="min",
)

trainer_temporal = pl.Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=max_epochs,
    gradient_clip_val=1.0,
    log_every_n_steps=50,
    enable_progress_bar=True,
    callbacks=[early_stop],
)

trainer_temporal.fit(model_temporal, train_dataloader, val_dataloader)
trainer_temporal.test(model_temporal, dataloaders=test_dataloader)

Epoch 8/199 ━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/390 0:00:00 • 0:00:04 100.94it/s v_num: 14664325.000 train_loss:   
                                                                                 1.907 val_loss: 18.126 val_acc:   
                                                                                 0.109                             

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3     ]
SLURM auto-requeueing enabled. Setting signal handlers.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.03171480819582939    │
│         test_loss         │     77.24649047851562     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 77.24649047851562, 'test_acc': 0.03171480819582939}]